# Day 4

## Tokenizing with code

In [ ]:
import tiktoken #herramienta para tokenizar texto, es decir, convertirlo en tokens que el modelo pueda entender.
# obtiene el tokenizador especifoco que usa el modelo gpt-4-1-mini
# encoding_for_model() devuelve un objeto de codificación que se puede usar para codificar y decodificar texto en tokens.
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [2]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]

In [3]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
6117 =  Ed
326 =  and
357 =  I
1299 =  like
9171 =  ban
26458 = offee
5148 =  pie


In [ ]:
encoding.decode([326])

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [7]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if not api_key:
    print("No GOOGLE_API_KEY was found - revisa tu archivo .env")
elif not api_key.startswith("AIza"):
    print("Se encontró una key, pero no empieza con 'AIza'. Verifica que sea una clave de Google AI Studio.")
else:
    print("GOOGLE_API_KEY encontrada y con formato válido ✅")

Se encontró una key, pero no empieza con 'AIza'. Verifica que sea una clave de Google AI Studio.


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

### A message to OpenAI is a list of dicts

In [16]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
]

response = openai.chat.completions.create(
    model="gemini-3.5-flash",  # Usa el modelo vigente
    messages=messages
)
response.choices[0].message.content

"Hi Ed! It's great to meet you. How can I help you today?"

### OK let's now ask a follow-up question

In [18]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [19]:
response = openai.chat.completions.create(
    model="gemini-3.5-flash",  # Usa el modelo vigente
    messages=messages
)
response.choices[0].message.content

"I don't know your name yet! Since I don't have access to your personal information, you'll have to tell me. \n\nWhat should I call you?"

### Wait, wha??

We just told you!

What's going on??

La cuestión es esta: cada llamada a un LLM carece totalmente de estado (*stateless*). Se trata de una llamada completamente nueva en cada ocasión. Como ingenieros de IA, es nuestra labor idear técnicas que den la impresión de que el LLM posee «memoria».

In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [21]:
response = openai.chat.completions.create(
    model="gemini-3.5-flash",  # Usa el modelo vigente
    messages=messages
)
response.choices[0].message.content

'Your name is Ed! How can I help you today?'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

